# Hafta 4 - Google Colab üzerinde CUDA destekli OpenCV kurulumu

Colab'ın hazır OpenCV paketi CUDA desteği olmadan derlenmiştir. Bu yüzden OpenCV, `opencv_contrib` içindeki CUDA modülleriyle birlikte kaynak koddan derlenir.

**Başlamadan önce:** Çalışma zamanı → Çalışma zamanı türünü değiştir → **T4 GPU**

Adımlar:
1. GPU ve CUDA sürümünü kontrol et
2. Gerekli paketleri kur
3. OpenCV ve opencv_contrib kaynak kodlarını indir
4. CUDA açık olarak CMake ile yapılandır
5. Derle ve kur (yaklaşık 25-40 dakika)
6. (İsteğe bağlı) Kurulumu Google Drive'a kaydet, bir sonraki oturumda derlemeden geri yükle
7. C++ ile CUDA testi: bir dizindeki görüntüleri GPU'da 1024x768'e boyutlandır

In [ ]:
# 1) GPU ve CUDA kontrolu
!nvidia-smi
!nvcc --version

In [ ]:
# 2) Derleme icin gerekli paketler
!apt-get -qq update
!apt-get -qq install -y build-essential cmake git pkg-config libjpeg-dev libpng-dev libtiff-dev

In [ ]:
# 3) Kaynak kodlar (iki depo da ayni surumde olmali)
%cd /content
!git clone -q --depth 1 --branch 4.14.0 https://github.com/opencv/opencv.git
!git clone -q --depth 1 --branch 4.14.0 https://github.com/opencv/opencv_contrib.git
!ls /content

In [ ]:
# 4) CMake yapilandirmasi
# WITH_CUDA=ON          : CUDA destegi
# CUDA_ARCH_BIN=7.5     : Colab T4 GPU'nun mimarisi (yalnizca bunun icin derlemek sureyi kisaltir)
# BUILD_LIST            : sadece gereken moduller derlenir (tam derleme 1.5 saati asabilir)
# OPENCV_GENERATE_PKGCONFIG=ON : C++ derlerken pkg-config ile kutuphaneler bulunur
!mkdir -p /content/opencv/build
%cd /content/opencv/build
!cmake .. \
  -D CMAKE_BUILD_TYPE=Release \
  -D CMAKE_INSTALL_PREFIX=/usr/local \
  -D OPENCV_EXTRA_MODULES_PATH=/content/opencv_contrib/modules \
  -D WITH_CUDA=ON \
  -D CUDA_ARCH_BIN=7.5 \
  -D CUDA_ARCH_PTX= \
  -D WITH_CUDNN=OFF \
  -D OPENCV_DNN_CUDA=OFF \
  -D BUILD_LIST=core,imgproc,imgcodecs,cudev,cudaarithm,cudaimgproc,cudawarping,cudafilters \
  -D BUILD_TESTS=OFF -D BUILD_PERF_TESTS=OFF -D BUILD_EXAMPLES=OFF -D BUILD_DOCS=OFF \
  -D OPENCV_GENERATE_PKGCONFIG=ON \
  2>&1 | tail -n 60

Yukarıdaki çıktıda **NVIDIA CUDA: YES** satırı ve *To be built* listesinde `cudaarithm cudafilters cudaimgproc cudawarping cudev` modülleri görünmelidir. Görünmüyorsa çalışma zamanının GPU olduğundan emin olun.

In [ ]:
# 5) Derleme ve kurulum (uzun surer)
!make -j$(nproc) 2>&1 | tail -n 5
!make install 2>&1 | tail -n 3
!ldconfig
!pkg-config --modversion opencv4

In [ ]:
# 6) (Istege bagli) Kurulumu Drive'a kaydet
from google.colab import drive
drive.mount('/content/drive')
!cd /usr/local && tar -czf /content/drive/MyDrive/opencv_cuda_kurulum.tar.gz lib/libopencv_* include/opencv4 lib/pkgconfig/opencv4.pc

# Bir sonraki oturumda derlemek yerine sadece sunu calistirin:
# !tar -xzf /content/drive/MyDrive/opencv_cuda_kurulum.tar.gz -C /usr/local && ldconfig

In [ ]:
# Test icin ornek goruntuler (OpenCV deposundaki ornekler)
%cd /content
!mkdir -p resimler
!cp opencv/samples/data/building.jpg opencv/samples/data/lena.jpg opencv/samples/data/fruits.jpg opencv/samples/data/baboon.jpg resimler/
!ls -la resimler

In [ ]:
%%writefile /content/cuda_boyutlandir.cpp
// CUDA destekli OpenCV testi:
// Bir dizindeki goruntuleri okuyup GPU uzerinde 1024x768'e boyutlandirir,
// sonucu CPU'daki resize ile karsilastirir.
#include <opencv2/opencv.hpp>
#include <opencv2/core/cuda.hpp>
#include <opencv2/cudawarping.hpp>
#include <iostream>
#include <filesystem>
#include <chrono>

using namespace cv;
using namespace std;
namespace fs = std::filesystem;

int main(int argc, char** argv)
{
    int gpuSayisi = cuda::getCudaEnabledDeviceCount();
    cout << "CUDA destekli GPU sayisi: " << gpuSayisi << endl;
    if (gpuSayisi == 0) { cout << "OpenCV CUDA destegi olmadan derlenmis!" << endl; return -1; }
    cuda::printShortCudaDeviceInfo(cuda::getDevice());

    fs::path giris = argc > 1 ? argv[1] : "resimler";
    fs::path cikis = argc > 2 ? argv[2] : "cikti_1024x768";
    fs::create_directories(cikis);

    for (const auto& oge : fs::directory_iterator(giris))
    {
        Mat resim = imread(oge.path().string());
        if (resim.empty()) continue;

        // GPU: yukle -> boyutlandir -> indir
        auto t1 = chrono::high_resolution_clock::now();
        cuda::GpuMat gGiris, gCikis;
        gGiris.upload(resim);
        cuda::resize(gGiris, gCikis, Size(1024, 768), 0, 0, INTER_LINEAR);
        Mat gpuSonuc;
        gCikis.download(gpuSonuc);
        auto t2 = chrono::high_resolution_clock::now();

        // CPU: ayni islem
        Mat cpuSonuc;
        resize(resim, cpuSonuc, Size(1024, 768), 0, 0, INTER_LINEAR);
        auto t3 = chrono::high_resolution_clock::now();

        imwrite((cikis / oge.path().filename()).string(), gpuSonuc);
        cout << oge.path().filename().string() << ": " << resim.cols << "x" << resim.rows
             << " -> " << gpuSonuc.cols << "x" << gpuSonuc.rows
             << " | GPU " << chrono::duration<double, milli>(t2 - t1).count() << " ms"
             << ", CPU " << chrono::duration<double, milli>(t3 - t2).count() << " ms"
             << " | GPU-CPU en buyuk fark: " << norm(gpuSonuc, cpuSonuc, NORM_INF) << endl;
    }
    return 0;
}

In [ ]:
# C++ programini derle ve calistir
%cd /content
!g++ -O2 -std=c++17 cuda_boyutlandir.cpp -o cuda_boyutlandir $(pkg-config --cflags --libs opencv4)
!./cuda_boyutlandir resimler cikti_1024x768
!ls -la cikti_1024x768

Notlar:
- İlk görüntüde GPU süresi, CUDA bağlamının başlatılması nedeniyle uzun çıkar. Sonraki görüntülerde süre düşer.
- GPU ve CPU `resize` sonuçları arasında 1-2 gri seviyelik farklar olabilir (yuvarlama farkı).